# 🚀 Chương 8: Kỹ Thuật Nâng Cao
## Prophet, Ensemble, Anomaly Detection
## Advanced Data Science - Session 5

---

**Mục tiêu chương này:**
- Sử dụng Facebook Prophet cho forecasting
- Ensemble Methods: kết hợp nhiều models
- Anomaly Detection trong Time Series
- Multi-step Forecasting strategies
- Best Practices & Pipeline hoàn chỉnh

## 8.1 Facebook Prophet

### 🎯 Prophet là gì?
- Thư viện forecasting của Facebook/Meta
- **Tự động** xử lý: trend, seasonality, holidays
- Rất dễ dùng, **không cần expertise** về time series
- Tốt với data có **nhiều seasonality** (daily, weekly, yearly)

### Ưu điểm:
- Robust với **missing data** và **outliers**
- Tự phát hiện **changepoints** (điểm thay đổi trend)
- Hỗ trợ thêm **holiday effects**
- API đơn giản: fit() → predict()

### Yêu cầu data:
- Column `ds`: datetime
- Column `y`: giá trị cần forecast

## 8.2 Ensemble Methods

### Ý tưởng: "Trí tuệ đám đông"
- 1 model có thể sai → Kết hợp nhiều models → Chính xác hơn

**Các cách kết hợp:**

| Phương pháp | Công thức | Khi nào dùng |
|-------------|-----------|---------------|
| **Simple Average** | (M1 + M2 + M3) / 3 | Các models tương đương |
| **Weighted Average** | w1·M1 + w2·M2 + w3·M3 | Biết model nào tốt hơn |
| **Stacking** | Meta-model học cách kết hợp | Có đủ data |

## 8.3 Anomaly Detection

### Phát hiện điểm bất thường:
- **Statistical:** Z-score, IQR, Rolling Mean ± k·Std
- **ML-based:** Isolation Forest, One-Class SVM
- **DL-based:** Autoencoder reconstruction error

---
## 🔬 Phần Thực Hành
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import IsolationForest
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

def evaluate(actual, predicted, model_name=''):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print(f'📊 {model_name:25s} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | MAPE: {mape:.2f}%')
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

print('✅ Import thành công!')

### 📝 Ví dụ 1: Facebook Prophet

In [ ]:
# Prophet cần install riêng: pip install prophet
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
    print('✅ Prophet đã được install')
except ImportError:
    PROPHET_AVAILABLE = False
    print('⚠️ Prophet chưa được install')
    print('   Chạy: pip install prophet')
    print('   Sẽ dùng code giả lập để minh họa')

# Tạo dữ liệu mẫu
np.random.seed(42)
n = 365 * 3
dates = pd.date_range('2021-01-01', periods=n, freq='D')
t = np.arange(n)

trend = 0.03 * t + 100
yearly = 20 * np.sin(2 * np.pi * t / 365)
weekly = 5 * np.sin(2 * np.pi * t / 7)
noise = np.random.normal(0, 5, n)
y = trend + yearly + weekly + noise

# Prophet cần format: ds (datetime), y (value)
df_prophet = pd.DataFrame({'ds': dates, 'y': y})

# Split
train_prophet = df_prophet[:-90]  # 3 tháng cuối để test
test_prophet = df_prophet[-90:]

if PROPHET_AVAILABLE:
    print('\n🔮 Training Prophet...')
    model_prophet = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05
    )
    model_prophet.fit(train_prophet)
    
    # Forecast
    future = model_prophet.make_future_dataframe(periods=90)
    forecast = model_prophet.predict(future)
    
    # Plot
    fig = model_prophet.plot(forecast)
    plt.title('📊 Prophet Forecast', fontweight='bold')
    plt.show()
    
    # Components
    fig2 = model_prophet.plot_components(forecast)
    plt.suptitle('📊 Prophet Components', fontweight='bold')
    plt.show()
    
    # Evaluate
    pred = forecast[-90:]['yhat'].values
    actual = test_prophet['y'].values
    evaluate(actual, pred, 'Prophet')
    
    print('\n💡 Prophet tự động phát hiện:')
    print('   - Trend (xu hướng tăng/giảm)')
    print('   - Yearly seasonality (mùa vụ theo năm)')
    print('   - Weekly seasonality (mùa vụ theo tuần)')
    print('   - Changepoints (điểm thay đổi trend)')
else:
    print('\n📋 Prophet Workflow (minh họa):')
    print('   1. df phải có columns: ds (datetime) và y (value)')
    print('   2. model = Prophet(yearly_seasonality=True)')
    print('   3. model.fit(train_df)')
    print('   4. future = model.make_future_dataframe(periods=90)')
    print('   5. forecast = model.predict(future)')
    print('   6. model.plot(forecast)  # Visualization')
    print('   7. model.plot_components(forecast)  # Decomposition')
    
    # Manual forecast for comparison later
    hw = ExponentialSmoothing(
        train_prophet.set_index('ds')['y'], 
        trend='add', seasonal='add', seasonal_periods=365
    ).fit()
    print('\n   (Dùng Holt-Winters làm baseline thay thế)')

### 📝 Ví dụ 2: Ensemble - Kết Hợp Nhiều Models

In [ ]:
print('='*60)
print('  ENSEMBLE METHODS')
print('='*60)

# Dữ liệu monthly
monthly = df_prophet.set_index('ds').resample('MS').mean()
train_m = monthly[:-6]
test_m = monthly[-6:]

# Model 1: Holt-Winters
hw = ExponentialSmoothing(train_m['y'], trend='add', seasonal='add', seasonal_periods=12).fit()
pred_hw = hw.forecast(6)

# Model 2: ARIMA
arima = ARIMA(train_m['y'], order=(1,1,1)).fit()
pred_arima = arima.forecast(6)

# Model 3: Simple baseline (last year same month)
pred_naive = train_m['y'].iloc[-12:-6].values

actual = test_m['y'].values

print('\n📊 Individual Models:')
r_hw = evaluate(actual, pred_hw, 'Holt-Winters')
r_arima = evaluate(actual, pred_arima, 'ARIMA(1,1,1)')
r_naive = evaluate(actual, pred_naive, 'Seasonal Naive')

# ===== ENSEMBLE =====
print('\n📊 Ensemble Methods:')

# Simple Average
pred_avg = (pred_hw.values + pred_arima.values + pred_naive) / 3
r_avg = evaluate(actual, pred_avg, 'Simple Average')

# Weighted Average (tỷ lệ nghịch với RMSE)
rmses = np.array([r_hw['RMSE'], r_arima['RMSE'], r_naive['RMSE']])
weights = (1 / rmses) / (1 / rmses).sum()  # Normalize
print(f'\n   Weights (auto): HW={weights[0]:.3f}, ARIMA={weights[1]:.3f}, Naive={weights[2]:.3f}')

pred_weighted = weights[0]*pred_hw.values + weights[1]*pred_arima.values + weights[2]*pred_naive
r_weighted = evaluate(actual, pred_weighted, 'Weighted Average')

# Median
pred_median = np.median([pred_hw.values, pred_arima.values, pred_naive], axis=0)
r_median = evaluate(actual, pred_median, 'Median Ensemble')

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_m.index, actual, 'ko-', label='Actual', markersize=8, linewidth=2)
ax.plot(test_m.index, pred_hw, 'b--s', label='Holt-Winters', alpha=0.5)
ax.plot(test_m.index, pred_arima, 'g--^', label='ARIMA', alpha=0.5)
ax.plot(test_m.index, pred_naive, 'orange', linestyle='--', marker='v', label='Naive', alpha=0.5)
ax.plot(test_m.index, pred_avg, 'r-D', label='Simple Avg', linewidth=2)
ax.plot(test_m.index, pred_weighted, 'm-*', label='Weighted Avg', linewidth=2, markersize=10)
ax.legend()
ax.set_title('📊 Ensemble: Kết Hợp Nhiều Models', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print('\n💡 Ensemble thường tốt hơn individual models!')
print('💡 Weighted Average cho trọng số cao hơn cho model tốt hơn')
print('💡 Median Ensemble robust với outliers')

### 📝 Ví dụ 3: Anomaly Detection - Statistical Methods

In [ ]:
print('='*60)
print('  ANOMALY DETECTION')
print('='*60)

# Tạo dữ liệu có anomaly
np.random.seed(42)
n = 500
dates = pd.date_range('2023-01-01', periods=n, freq='D')
t = np.arange(n)

normal = 100 + 10 * np.sin(2 * np.pi * t / 30) + np.random.normal(0, 3, n)

# Chèn anomalies
anomaly_indices = [50, 120, 200, 300, 350, 420]
anomaly_values = [60, 160, 55, 170, 50, 155]
for idx, val in zip(anomaly_indices, anomaly_values):
    normal[idx] = val

data_anom = pd.Series(normal, index=dates)

# ===== Method 1: Z-Score =====
print('\n📌 Method 1: Z-Score')
z_scores = (data_anom - data_anom.mean()) / data_anom.std()
z_anomalies = abs(z_scores) > 3
print(f'   Threshold: |Z| > 3')
print(f'   Anomalies found: {z_anomalies.sum()}')

# ===== Method 2: Rolling Mean ± k*Std =====
print('\n📌 Method 2: Rolling Mean ± k·Std')
window = 30
k = 2.5
rolling_mean = data_anom.rolling(window, center=True).mean()
rolling_std = data_anom.rolling(window, center=True).std()
upper = rolling_mean + k * rolling_std
lower = rolling_mean - k * rolling_std
rolling_anomalies = (data_anom > upper) | (data_anom < lower)
print(f'   Window: {window}, k: {k}')
print(f'   Anomalies found: {rolling_anomalies.sum()}')

# ===== Method 3: IQR =====
print('\n📌 Method 3: IQR (Interquartile Range)')
Q1 = data_anom.quantile(0.25)
Q3 = data_anom.quantile(0.75)
IQR = Q3 - Q1
iqr_anomalies = (data_anom < Q1 - 1.5*IQR) | (data_anom > Q3 + 1.5*IQR)
print(f'   Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
print(f'   Range: [{Q1-1.5*IQR:.2f}, {Q3+1.5*IQR:.2f}]')
print(f'   Anomalies found: {iqr_anomalies.sum()}')

# Plot all methods
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Z-Score
axes[0].plot(data_anom.index, data_anom, 'b-', alpha=0.5, label='Data')
axes[0].scatter(data_anom.index[z_anomalies], data_anom[z_anomalies], 
               c='red', s=100, zorder=5, label=f'Anomalies ({z_anomalies.sum()})')
axes[0].set_title('📊 Method 1: Z-Score (|Z| > 3)', fontweight='bold')
axes[0].legend()

# Rolling
axes[1].plot(data_anom.index, data_anom, 'b-', alpha=0.5, label='Data')
axes[1].plot(data_anom.index, rolling_mean, 'g-', label='Rolling Mean')
axes[1].fill_between(data_anom.index, lower, upper, alpha=0.2, color='green', label='Normal Range')
axes[1].scatter(data_anom.index[rolling_anomalies], data_anom[rolling_anomalies], 
               c='red', s=100, zorder=5, label=f'Anomalies ({rolling_anomalies.sum()})')
axes[1].set_title('📊 Method 2: Rolling Mean ± k·Std', fontweight='bold')
axes[1].legend()

# IQR
axes[2].plot(data_anom.index, data_anom, 'b-', alpha=0.5, label='Data')
axes[2].axhline(y=Q3 + 1.5*IQR, color='red', linestyle='--', alpha=0.7, label='Upper/Lower bound')
axes[2].axhline(y=Q1 - 1.5*IQR, color='red', linestyle='--', alpha=0.7)
axes[2].scatter(data_anom.index[iqr_anomalies], data_anom[iqr_anomalies], 
               c='red', s=100, zorder=5, label=f'Anomalies ({iqr_anomalies.sum()})')
axes[2].set_title('📊 Method 3: IQR', fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.show()

print('\n💡 Rolling Mean ± k·Std thường tốt nhất cho Time Series')
print('   vì nó thích ứng theo thời gian (adaptive)')

### 📝 Ví dụ 4: Anomaly Detection - Isolation Forest

In [ ]:
# ===== Isolation Forest =====
print('='*60)
print('  ISOLATION FOREST')
print('='*60)

# Tạo features cho Isolation Forest
df_iso = pd.DataFrame({
    'value': data_anom.values,
    'rolling_mean_7': data_anom.rolling(7).mean().values,
    'rolling_std_7': data_anom.rolling(7).std().values,
    'diff': data_anom.diff().values,
    'lag_1': data_anom.shift(1).values,
}, index=data_anom.index)

df_iso = df_iso.dropna()

# Fit Isolation Forest
iso_forest = IsolationForest(
    contamination=0.02,  # Kỳ vọng 2% data là anomaly
    random_state=42,
    n_estimators=200
)

df_iso['anomaly'] = iso_forest.fit_predict(df_iso)
df_iso['anomaly_score'] = iso_forest.decision_function(df_iso.drop(columns=['anomaly']))

anomalies_iso = df_iso[df_iso['anomaly'] == -1]
print(f'\n📊 Anomalies detected: {len(anomalies_iso)}')
print(f'   Contamination rate: 2%')

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df_iso.index, df_iso['value'], 'b-', alpha=0.5)
axes[0].scatter(anomalies_iso.index, anomalies_iso['value'], 
               c='red', s=100, zorder=5, label=f'Anomalies ({len(anomalies_iso)})')
axes[0].set_title('📊 Isolation Forest: Detected Anomalies', fontweight='bold')
axes[0].legend()

axes[1].plot(df_iso.index, df_iso['anomaly_score'], 'g-', alpha=0.7)
axes[1].axhline(y=0, color='red', linestyle='--', label='Threshold')
axes[1].fill_between(df_iso.index, df_iso['anomaly_score'], 0, 
                    where=df_iso['anomaly_score'] < 0, alpha=0.3, color='red')
axes[1].set_title('📊 Anomaly Score (< 0 = anomaly)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\n💡 Isolation Forest: ML-based anomaly detection')
print('   - Không cần biết distribution của data')
print('   - "Isolate" anomalies bằng random splits')
print('   - Anomalies bị isolate nhanh hơn → Score thấp hơn')

### 📝 Ví dụ 5: Complete Pipeline - Best Practices

In [ ]:
print('='*60)
print('  COMPLETE TIME SERIES PIPELINE')
print('='*60)

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.ensemble import RandomForestRegressor

# Tạo dữ liệu phức tạp
np.random.seed(42)
n = 730
dates = pd.date_range('2022-01-01', periods=n, freq='D')
t = np.arange(n)

trend = 0.05 * t + 100
yearly = 20 * np.sin(2 * np.pi * t / 365)
weekly = 5 * np.sin(2 * np.pi * t / 7)
noise = np.random.normal(0, 3, n)
sales = trend + yearly + weekly + noise

# Thêm một vài anomalies
sales[100] = 30  # Drop
sales[300] = 250  # Spike

df = pd.DataFrame({'sales': sales}, index=dates)

print('\n📌 Step 1: EDA (Exploratory Data Analysis)')
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0,0].plot(df.index, df['sales'])
axes[0,0].set_title('Original Data')
axes[0,1].hist(df['sales'], bins=50, edgecolor='black')
axes[0,1].set_title('Distribution')
axes[1,0].plot(df.index, df['sales'].rolling(30).mean())
axes[1,0].set_title('30-Day Rolling Mean')
axes[1,0].fill_between(df.index, 
                       df['sales'].rolling(30).mean() - df['sales'].rolling(30).std(),
                       df['sales'].rolling(30).mean() + df['sales'].rolling(30).std(),
                       alpha=0.2)
decomp = seasonal_decompose(df['sales'], period=365, model='additive')
decomp.seasonal[:365].plot(ax=axes[1,1])
axes[1,1].set_title('Seasonal Component (1 year)')
plt.suptitle('📊 Step 1: EDA', fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📌 Step 2: Anomaly Detection & Handling')
rolling_mean = df['sales'].rolling(30, center=True).mean()
rolling_std = df['sales'].rolling(30, center=True).std()
anomalies = (df['sales'] > rolling_mean + 3*rolling_std) | (df['sales'] < rolling_mean - 3*rolling_std)
print(f'   Anomalies found: {anomalies.sum()}')
# Replace anomalies with rolling mean
df_clean = df.copy()
df_clean.loc[anomalies, 'sales'] = rolling_mean[anomalies]

print('\n📌 Step 3: Stationarity Check')
p_val = adfuller(df_clean['sales'].dropna())[1]
print(f'   ADF p-value: {p_val:.4f}', '✅ Stationary' if p_val < 0.05 else '❌ Non-Stationary')

print('\n📌 Step 4: Train/Test Split')
train = df_clean[:-90]
test = df_clean[-90:]
print(f'   Train: {len(train)} days')
print(f'   Test: {len(test)} days')

print('\n📌 Step 5: Model Training & Comparison')

# Model 1: Holt-Winters
hw = ExponentialSmoothing(train['sales'], trend='add', seasonal='add', seasonal_periods=365).fit()
pred_hw = hw.forecast(90)

# Model 2: ARIMA
arima = ARIMA(train['sales'], order=(2,1,1)).fit()
pred_arima = arima.forecast(90)

# Model 3: Random Forest
def create_ml_features(df, target='sales'):
    data = df.copy()
    for lag in [1,7,14,30]:
        data[f'lag_{lag}'] = data[target].shift(lag)
    data['rolling_mean_7'] = data[target].shift(1).rolling(7).mean()
    data['rolling_mean_30'] = data[target].shift(1).rolling(30).mean()
    data['day_of_week'] = data.index.dayofweek
    data['month'] = data.index.month
    data['day_of_year'] = data.index.dayofyear
    return data.dropna()

df_ml = create_ml_features(df_clean)
features = [c for c in df_ml.columns if c != 'sales']

split_date = test.index[0]
X_tr = df_ml.loc[:split_date].iloc[:-1][features]
y_tr = df_ml.loc[:split_date].iloc[:-1]['sales']

# Recursive prediction cho RF
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_tr, y_tr)

# One-shot predict cho simplicity
X_te = df_ml.loc[split_date:][features].head(90)
if len(X_te) > 0:
    pred_rf = rf.predict(X_te)
else:
    pred_rf = pred_hw.values  # Fallback

actual = test['sales'].values[:len(pred_rf)]

print('\n📊 Individual Results:')
r_hw = evaluate(actual, pred_hw.values[:len(actual)], 'Holt-Winters')
r_arima = evaluate(actual, pred_arima.values[:len(actual)], 'ARIMA(2,1,1)')
if len(X_te) > 0:
    r_rf = evaluate(actual, pred_rf[:len(actual)], 'Random Forest')

# Ensemble
print('\n📊 Ensemble:')
pred_ensemble = (pred_hw.values[:len(actual)] + pred_arima.values[:len(actual)]) / 2
if len(X_te) > 0:
    pred_ensemble = (pred_hw.values[:len(actual)] + pred_arima.values[:len(actual)] + pred_rf[:len(actual)]) / 3
r_ens = evaluate(actual, pred_ensemble, 'Ensemble Average')

# Final plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index[-180:], train['sales'][-180:], label='Train', color='blue', alpha=0.5)
ax.plot(test.index[:len(actual)], actual, label='Actual', color='black', linewidth=2)
ax.plot(test.index[:len(actual)], pred_hw.values[:len(actual)], '--', label='Holt-Winters', alpha=0.7)
ax.plot(test.index[:len(actual)], pred_arima.values[:len(actual)], '--', label='ARIMA', alpha=0.7)
ax.plot(test.index[:len(actual)], pred_ensemble, 'r-', label='Ensemble ✅', linewidth=2)
ax.axvline(x=test.index[0], color='gray', linestyle=':', alpha=0.5)
ax.legend()
ax.set_title('📊 Complete Pipeline: Final Forecast', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print('\n' + '='*60)
print('  BEST PRACTICES CHECKLIST')
print('='*60)
print('  ✅ 1. EDA: Hiểu dữ liệu trước khi model')
print('  ✅ 2. Anomaly Detection: Phát hiện & xử lý outliers')
print('  ✅ 3. Stationarity: Check & transform nếu cần')
print('  ✅ 4. Train/Test Split: THEO THỜI GIAN, không random!')
print('  ✅ 5. Multiple Models: Thử nhiều approaches')
print('  ✅ 6. Ensemble: Kết hợp cho kết quả tốt hơn')
print('  ✅ 7. Evaluation: MAE, RMSE, MAPE')
print('  ✅ 8. Visualization: Luôn plot để kiểm tra')

---
## 🏋️ BÀI TẬP THỰC HÀNH
---

### Bài 1: Prophet Forecasting (⭐ Dễ)

1. Install Prophet: `pip install prophet`
2. Đọc dữ liệu `retail_sales_dataset.csv`
3. Chuyển sang format Prophet (ds, y)
4. Fit Prophet, forecast 30 ngày
5. Plot forecast và components

In [ ]:
# TODO: Viết code ở đây

### Bài 2: Ensemble (⭐⭐ Trung bình)

1. Dùng `stores_sales_forecasting.csv`
2. Fit ít nhất 3 models: ARIMA, Holt-Winters, Random Forest
3. Tạo Ensemble: Simple Average, Weighted Average
4. So sánh tất cả methods
5. Phân tích: Ensemble có tốt hơn không?

In [ ]:
# TODO: Viết code ở đây

### Bài 3: Complete Pipeline (⭐⭐⭐ Nâng cao)

Thực hiện pipeline hoàn chỉnh trên dữ liệu thực:
1. EDA đầy đủ
2. Anomaly Detection (dùng 2 phương pháp)
3. Stationarity check + transform
4. Fit tất cả models đã học (ARIMA, SARIMA, Holt-Winters, RF, LSTM)
5. Ensemble
6. Đánh giá tổng hợp
7. Viết kết luận: Model/Pipeline nào tốt nhất? Tại sao?

In [ ]:
# TODO: Viết code ở đây